# Cyber-manipulation Susceptibility Pipeline: Demo and Stage Guide

This notebook explains the end-to-end pipeline that simulates and analyses
inter-individual differences in susceptibility to cyber-manipulation of political
opinions. It is meant as a clear, professional walkthrough: what each stage does,
how to run it, and how to load and read the outputs.

The study models three independent ontological state spaces and crosses them into
scenarios:

1. **Profiles**: high-resolution synthetic persons (demographics, Big Five, a full
   political-psychology battery, and more). A curated reduction keeps the
   research-relevant core so the simulating agent can attribute cleanly and the
   moderator models stay well-posed.
2. **Opinions**: an issue-position taxonomy of political stances, grouped into 7
   parent clusters (issue domains), each with directional leaves and a per-leaf
   adversarial direction.
3. **Attacks**: DISARM-red Plan / Prepare / Execute operation triplets.

A scenario pairs one profile, one opinion cluster, and one attack triplet. A
language-model agent rates the profile's opinion on every leaf before and after the
attack; the signed, direction-aware change is the adversarial effectivity.


## 1. Repository layout

```text
src/backend/pipeline/
  separate/        numbered stages 01..09 (plus the optional network stages 01b/02b/04b/05b)
  full/            run_full_pipeline.py orchestrator
src/backend/utils/
  analysis/conditional_susceptibility.py   the susceptibility estimators
  figures/individual_layer_paper_figures.py paper-ready PNGs
scripts/
  tests/           run_1.sh .. run_5.sh (small validation runs)
  production/      run_1.sh (the individual-layer production run)
evaluation/
  tests/run_N/     test-run outputs
  production/run_1/ production outputs
```


## 2. The pipeline stages

The orchestrator `run_full_pipeline.py` runs the numbered stages in order. The
network stages (suffix b) only run when `--with-network-exposure` is set; the
individual-layer production run leaves them off.

| Stage | Name | What it does |
|-------|------|--------------|
| 01 | create_scenarios | Select scenarios from the 10K integrated set (stratified or max-entropy), map each into a reduced profile + DISARM triplet + opinion cluster |
| 02 | assess_baseline_opinions | Agent rates the private baseline opinion B on every leaf (one call per scenario, cluster at once) |
| 03 | run_opinion_attacks | Compile the DISARM triplet into a deterministic attack-vector spec |
| 04 | assess_post_attack_opinions | Agent rates the private post-attack opinion P on every leaf |
| 05 | compute_effectivity_deltas | Build the per-leaf deltas: delta = P - B, adversarial_effectivity = (P - B) * d; emit the long and wide analysis tables |
| 06 | construct_structural_equation_model | Multilevel moderation, the conditional susceptibility index, and the block-wise family model |
| 07 | generate_research_visuals | Interactive HTML dashboard plus paper-ready individual-layer PNGs |
| 08 | generate_publication_assets | Static figure and asset export |

The optional network stages add a baseline-network state BN (02b) and a
post-attack-network state PN (04b) using an empirical exposure graph, plus the
network analysis (05b). They are out of scope for the individual-layer production run.


## 3. How to run

Small validation runs live in `scripts/tests/`. The individual-layer production run
is `scripts/production/run_1.sh` (1,000 scenarios, all 7 domains, no network, reduced
profile, max-entropy sub-sampling).

```bash
# individual-layer production run (about 2,000 LLM calls)
bash scripts/production/run_1.sh --verbose

# a fast single-domain validation run
bash scripts/tests/run_5.sh
```

Both launchers read `OPENROUTER_API_KEY` from `.env` and check the projected budget
before spending. The example cells below read an already-completed run.


In [ ]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# Point this at any completed run directory.
RUN = Path('evaluation/tests/run_5')
STAGE05 = RUN / 'stage_outputs' / '05_compute_effectivity_deltas'
SEM = pd.read_csv(STAGE05 / 'sem_long_encoded.csv')
print('rows:', len(SEM), '| profiles:', SEM['profile_id'].nunique(), '| opinion leaves:', SEM['opinion_leaf'].nunique())


## 4. Stage 01: scenario construction and the reduced profile

Stage 01 consumes the pre-built 10,000-row integrated set. Two production-relevant
controls:

- `--max-entropy-subsample` draws the requested number of scenarios to maximise the
  entropy of the (issue domain x Execute tactic) grid, preserving source diversity.
- `--profile-skip-subtrees` (with a curated default) drops about 40 percent of the
  profile: the redundant HEXACO, Eysenck and Hexad personality taxonomies and the
  low-relevance goals, values, safety, criminal and administrative subtrees. The
  research core stays: demographics, socioeconomics, Big Five, the political-psychology
  battery, moral foundations, religion, and digital literacy.

The sampling provenance is written to the stage audit.


In [ ]:
aud = json.load(open(RUN / 'stage_outputs' / '01_create_scenarios' / 'scenario_compatibility_audit.json'))
prov = aud.get('sampling_provenance', {})
print('sampler:', prov.get('sampler'))
print('profile features kept:', prov.get('n_profile_features_kept'))
print('dropped subtrees:', prov.get('profile_skip_subtrees'))


## 5. Stages 02 to 05: measurement and the deltas

These four stages turn each scenario into measured susceptibility data.

| Stage | Symbol | What happens |
|-------|--------|--------------|
| 02 assess_baseline_opinions | B | The persona privately rates every opinion leaf in its assigned issue cluster, with no attack present. |
| 03 run_opinion_attacks | (spec) | The DISARM Plan / Prepare / Execute triplet is compiled into one deterministic attack-vector specification for the scenario. |
| 04 assess_post_attack_opinions | P | The same persona is exposed to that attack and privately re-rates the same leaves. |
| 05 compute_effectivity_deltas | AE | B and P are combined into the per-leaf effectivity table (long and wide). |

The core identity, recomputed and checked on every run, is

```text
adversarial_effectivity = (P - B) * d        d in {-1, +1} per leaf
```

where `d` is the per-leaf adversarial direction: `+1` where the attacker wants to
amplify the opinion and `-1` where the attacker wants to erode it. A positive
effectivity therefore always means the opinion moved the way the attacker intended,
whatever the sign of the raw change. The direction is set from the opinion ontology,
not inferred from the result.

### Unit of analysis

A scenario is one (persona x attack x opinion cluster) cell. It contributes many
opinion-leaf measurements that are nested within it and are not independent of one
another. In `sem_long` the `scenario_id` is encoded as `scenario_<id>__<leaf>`, so
the leaf suffix must be stripped to recover the scenario. The inferential tests in
stage 06 aggregate to the scenario mean first, so each scenario is one independent
observation when attacks, tactics or domains are compared; the figures still show the
full leaf-level cloud but draw their mean intervals with a scenario-clustered
bootstrap for the same reason.

### What is saved

Every per-scenario, per-leaf delta is retained for post-hoc analysis:
`effectivity_deltas.jsonl` (each record also carries the full source profile),
`sem_long_raw.csv` (human-readable) and `sem_long_encoded.csv` (model-ready). The
cell below loads the long table, verifies the identity exactly, and reports the
headline effect at the correct scenario unit.


In [ ]:
ae = SEM.dropna(subset=['adversarial_effectivity']).copy()

# True scenario = one (persona x attack x cluster) cell; sem_long encodes it as
# 'scenario_<id>__<leaf>', so strip the leaf suffix to recover the independent unit.
ae['scenario'] = ae['scenario_id'].str.split('__', n=1).str[0]
n_leaf, n_scn = len(ae), ae['scenario'].nunique()
print(f"leaf-level measurements: {n_leaf}   independent scenarios: {n_scn}   "
      f"(~{n_leaf / max(n_scn, 1):.1f} leaves per scenario)")

# The effectivity identity must hold exactly on every row.
identity_err = ((ae['post_score'] - ae['baseline_score']) * ae['adversarial_direction']
                - ae['adversarial_effectivity']).abs().max()
print(f"max identity error |AE - (P-B)*d|: {identity_err:.3g}  (0.0 = exact)")

# Headline effect at the correct (scenario) unit, not the inflated leaf count.
scn_mean = ae.groupby('scenario')['adversarial_effectivity'].mean()
print(f"mean effectivity (scenario unit): {scn_mean.mean():+.2f}   "
      f"scenarios moved toward goal: {(scn_mean > 0).mean() * 100:.1f}%")

# Inter-individual heterogeneity, and the amplify (+1) vs erode (-1) split.
print(f"between-profile SD: {ae.groupby('profile_id')['adversarial_effectivity'].mean().std():.2f}")
print("mean effectivity by adversarial direction d:")
print(ae.groupby('adversarial_direction')['adversarial_effectivity'].mean().round(2).to_string())


## 6. Stage 06: multilevel moderation and the susceptibility estimators

Stage 06 answers which profile characteristics moderate susceptibility. It runs
several complementary models:

- A curated multivariate OLS (Big Five plus demographics) with cluster-robust
  bootstrap confidence intervals: the interpretable primary moderators.
- A hierarchical leave-one-group-out variance decomposition: which ontology family
  carries unique explanatory power.
- The Conditional Susceptibility Index per profile, with both a pooled task and
  per-DISARM-Execute-tactic tasks, so the dashboard can be scoped to a specific
  attack vector.
- The block-wise family model (next section).


In [ ]:
SEM06 = RUN / 'analysis' / 'sem'
ols = pd.read_csv(SEM06 / 'ols_robust_params.csv')
ols = ols[ols['term'] != 'Intercept'].copy()
ols['significant'] = (ols['conf_low'] > 0) | (ols['conf_high'] < 0)
cols = ['term', 'estimate', 'p_value', 'conf_low', 'conf_high', 'significant']
ols.sort_values('estimate', ascending=False)[cols].round(3)


## 7. The scalable block-wise family model

A single ridge over the full profile (hundreds of features) overfits, especially at
small samples. The block-wise model fixes this and stays interpretable:

1. Group the profile features into ontology families (Big Five, the political
   inventories, demographics, and so on).
2. Fit a regularized sub-model per family on the profile-level mean effect.
3. Combine the sub-models with a heavily-regularized meta-learner.
4. Evaluate everything with nested cross-validation, so the reported R2 has no
   stacking leakage and degrades gracefully toward zero (rather than to large
   negative values) when the trait signal is weak.

Because no sub-model ever sees the full feature space, the conditioning is
independent of the total feature count: it scales to the entire profile ontology.
The per-family standalone out-of-fold R2 shows where signal concentrates.


In [ ]:
bw_path = SEM06 / 'blockwise_family_susceptibility.csv'
if bw_path.exists():
    bw = pd.read_csv(bw_path)
    display(bw.sort_values('standalone_oof_r2', ascending=False))
else:
    print('Run a fresh run (the block-wise model is produced by stage 06).')


## 8. Conditional susceptibility index and per-attack scoping

The index ranks profiles by predicted effectivity under a chosen target set. Because
stage 06 fits per-Execute-tactic tasks alongside the pooled task, the dashboard
estimator can be scoped to a specific attack vector (Conduct Pump Priming, Deliver
Content, Maximise Exposure, and so on) rather than only the pooled view.


In [ ]:
idx = pd.read_csv(SEM06 / 'profile_susceptibility_index.csv')
keep = [c for c in idx.columns if c in ('profile_id', 'susceptibility_index_pct', 'mean_adversarial_eff')]
idx[keep].head(8) if keep else idx.head(8)


## 9. Paper-ready figures

Stage 07 writes high-resolution PNGs (and one interactive HTML) to
`visuals/paper_figures/`, complementary to the interactive HTML dashboard. They are
dense but readable, and they respect the sparse sampling of the full ontological
state space (thinly-observed cells are masked rather than imputed as zero):

- `individual_layer_overview.png`: a coherent multipanel (effect distribution,
  curated moderators, block-wise family signal, most-movable leaves).
- `attack_opinion_effectiveness_clustermap.png`: opinion leaf by Execute tactic
  effectiveness with dendrograms mounted on both axes and an issue-domain colour strip.
- `disarm_attack_hierarchy_effectiveness.png`: effectiveness at multiple DISARM levels
  (Plan, Prepare and Execute tactics, and the operation complexity tier), not only the
  high-level nodes.
- `moderator_by_opinion_domain.png`: which trait constructs moderate susceptibility,
  per opinion domain, with a row dendrogram.
- `susceptibility_structure.png`: baseline to post movement, effect by adversarial
  direction, inter-individual heterogeneity, and the complexity-tier dose-response.
- `opinion_leaf_susceptibility_by_domain.png`, `execute_tactic_effectiveness.png`,
  `blockwise_family_susceptibility.png`, and `attack_opinion_effectiveness_interactive.html`.

You can also regenerate them directly from a finished run:


In [ ]:
from src.backend.utils.figures.individual_layer_paper_figures import generate_individual_layer_paper_figures
files = generate_individual_layer_paper_figures(
    sem_long_csv_path=str(STAGE05 / 'sem_long_encoded.csv'),
    stage06_dir=str(RUN / 'analysis' / 'sem'),
    output_dir=str(RUN / 'visuals'),
    run_id=RUN.name,
)
for f in files:
    print(Path(f).name)


## 10. Reproducibility and data retention

Every run keeps the full provenance: each raw LLM request and response under
`provenance/raw_llm/`, every per-scenario and per-leaf delta under `stage_outputs/`,
and the run configuration under `config/`. This is what supports post-hoc analysis
beyond the built-in moderation. The individual-layer production run lives at
`evaluation/production/run_1/` and is launched with `scripts/production/run_1.sh`.
